In [1]:
import os
import zipfile
import pandas as pd
import re
import shutil
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# === CONFIGURATION ===
ZIP_FILE = "Arunachal Pradesh_all_BOQ_live.zip"  # your input zip
EXTRACT_DIR = "Arunachal Pradesh_all_BOQ_live"
OUTPUT_FILE = "Arunachal Pradesh_all_BOQ_live_Summary.xlsx"

# === Helper Functions ===

def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_class(text):
    text = str(text).lower()
    if 'k-9' in text or 'k9' in text:
        return 'DI-K9'
    elif 'k-7' in text:
        return 'DI-K7'
    elif 'm.s' in text or 'ms' in text:
        return 'MS'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'cpvc' in text:
        return 'CPVC'
    elif 'pvc' in text:
        return 'PVC'
    return None

def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

# === STEP 1: Extract the ZIP ===
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print(f"✅ Extracted: {ZIP_FILE} to {EXTRACT_DIR}")

# === STEP 2: Locate BOQ files ===
root_folder = os.path.join(EXTRACT_DIR, os.listdir(EXTRACT_DIR)[0])
results = []

for tdr_folder in os.listdir(root_folder):
    tdr_path = os.path.join(root_folder, tdr_folder)
    if not os.path.isdir(tdr_path):
        continue
    for file in os.listdir(tdr_path):
        if file.lower().endswith(('.xls', '.xlsx')):
            file_path = os.path.join(tdr_path, file)
            try:
                df = pd.read_excel(file_path, header=2, engine='openpyxl' if file.endswith('.xlsx') else 'xlrd')
                df.columns = df.columns.map(str)

                # Rename to known columns
                if len(df.columns) >= 5:
                    df = df.rename(columns={
                        df.columns[1]: "Item Description",
                        df.columns[2]: "Units",
                        df.columns[3]: "Estimate Rate",
                        df.columns[4]: "Quantity"
                    })

                    # Clean & enrich
                    df["class"] = df["Item Description"].apply(extract_class)
                    df["DIA"] = df["Item Description"].apply(extract_dia)
                    df["Units"] = df["Units"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
                    df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')
                    df["Estimate Rate"] = pd.to_numeric(df["Estimate Rate"], errors='coerce')

                    # Filter rows for DI Pipe or related items
                    df_pipe = df[df["Item Description"].str.contains("pipe|di|k-9|k-7|m.s|hdpe|pvc|cpvc", case=False, na=False)]

                    for _, row in df_pipe.iterrows():
                        results.append({
                            "TDR": tdr_folder,
                            "BOQ File": file,
                            "Item Description": clean_illegal_chars(row["Item Description"]),
                            "DI-K9": "Yes" if row["class"] == "DI-K9" else "",
                            "DI-K7": "Yes" if row["class"] == "DI-K7" else "",
                            "MS": "Yes" if row["class"] == "MS" else "",
                            "HDPE": "Yes" if row["class"] == "HDPE" else "",
                            "CPVC": "Yes" if row["class"] == "CPVC" else "",
                            "PVC": "Yes" if row["class"] == "PVC" else "",
                            "DIA": row["DIA"],
                            "Estimate Rate": row["Estimate Rate"],
                            "Units": row["Units"],
                            "Quantity": row["Quantity"]
                        })
            except Exception as e:
                results.append({
                    "TDR": tdr_folder,
                    "BOQ File": file,
                    "Item Description": f"❌ Error reading file: {e}"
                })

# === STEP 3: Save to Excel ===
df_summary = pd.DataFrame(results)
df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
df_summary.to_excel(OUTPUT_FILE, index=False)

print(f"🏁 Done! DI Pipe BOQ data saved to: {OUTPUT_FILE}")


✅ Extracted: Arunachal Pradesh_all_BOQ_live.zip to Arunachal Pradesh_all_BOQ_live
🏁 Done! DI Pipe BOQ data saved to: Arunachal Pradesh_all_BOQ_live_Summary.xlsx
